In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [2]:
df = pd.read_csv('../../data/olist/olist_orders_dataset.csv')
df_review = pd.read_csv('../../data/olist/olist_order_reviews_dataset.csv')
df_customer = pd.read_csv('../../data/olist/olist_customers_dataset.csv')

In [3]:
df['is_delivered'] = (df['order_status'] == 'delivered').astype(int)

df_review['has_review'] = df_review['review_comment_message'].notna().astype(int)

df_review_slim = df_review[['order_id', 'has_review']]

In [4]:
df_order_review = df.merge(df_review_slim, on='order_id', how='left')
df_order_review['has_review'] = df_order_review['has_review'].fillna(0).astype(int)

In [5]:
customer_funnel = df_order_review.groupby('customer_id').agg(
    has_order=('order_id', 'count'),          # 有下单
    has_delivered=('is_delivered', 'max'),     # 至少1单送达
    has_review=('has_review', 'max')           # 至少1条评论
).reset_index()
customer_funnel['has_order'] = 1

In [6]:
step_order = customer_funnel['has_order'].sum()
step_deliver = customer_funnel['has_delivered'].sum()
step_review = customer_funnel['has_review'].sum()

funnel = pd.DataFrame({
    'stage': ['下单客户', '已送达客户', '有评论客户'],
    'cnt': [step_order, step_deliver, step_review]
})

# 计算转化率
funnel['prev_cnt'] = funnel['cnt'].shift(1)
funnel['step_conv'] = funnel['cnt'] / funnel['prev_cnt']
funnel['total_conv'] = funnel['cnt'] / funnel['cnt'].iloc[0]

In [7]:
print("===== 客户行为漏斗 =====")
print(funnel)

# 画漏斗图
fig = px.funnel(funnel, x='cnt', y='stage', title='Olist 客户转化漏斗')
fig.show()

===== 客户行为漏斗 =====
   stage    cnt  prev_cnt  step_conv  total_conv
0   下单客户  99441       NaN        NaN    1.000000
1  已送达客户  96478   99441.0   0.970203    0.970203
2  有评论客户  40836   96478.0   0.423267    0.410656
